# nb_metadata_sqldb_prototype — metadata framework on Fabric SQL Database
**Working prototype.** The DDL and every statement below execute here against a local SQLite
equivalent (so the SQL is proven and the notebook runs anywhere), and switch to **Fabric SQL
Database** via `TARGET="fabric"` with `pyodbc` + Entra token. Section 27 of the internals doc
covers the gotchas; this notebook is their executable counterpart.

**Design rules demonstrated:** parameterized SQL only · one config fetch per run, cached ·
batched run-log writes · forward-only watermark MERGE · secrets by name, resolved at run time ·
the SQL DB touched twice per run, not once per row.

In [1]:
NOTEBOOK_NAME = "nb_metadata_sqldb_prototype"
TARGET        = "local"          # "local" (SQLite demo) | "fabric" (Fabric SQL Database)
SQL_SERVER    = "<your-sqldb>.database.fabric.microsoft.com"   # Fabric SQL DB connection string host
SQL_DATABASE  = "etl_metadata"
LOCAL_DB      = "/tmp/etl_metadata_demo.db"

In [2]:
import json, os, time, uuid
from datetime import datetime, timezone, date

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def connect():
    """Two paths, one interface. Fabric: pyodbc + Entra token (see gotchas below).
    Local: sqlite3 with the same table shapes so every statement is exercised."""
    if TARGET == "fabric":
        import pyodbc, struct, notebookutils
        # GOTCHA 1: the token must be UTF-16-LE and length-prefixed into attrs_before[1256].
        token = notebookutils.credentials.getToken("https://database.windows.net/").encode("utf-16-le")
        tok = struct.pack(f"<I{len(token)}s", len(token), token)
        # GOTCHA 2: driver name varies by runtime image - probe, do not hard-code.
        drivers = [d for d in pyodbc.drivers() if "ODBC Driver" in d and "SQL Server" in d]
        if not drivers:
            raise RuntimeError("No SQL Server ODBC driver found in this runtime image.")
        driver = sorted(drivers)[-1]
        cs = (f"Driver={{{driver}}};Server={SQL_SERVER},1433;Database={SQL_DATABASE};"
              f"Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;")
        return pyodbc.connect(cs, attrs_before={1256: tok})   # SQL_COPT_SS_ACCESS_TOKEN
    import sqlite3
    con = sqlite3.connect(LOCAL_DB)
    con.execute("PRAGMA foreign_keys = ON")
    return con

PARAM = "?"    # both pyodbc and sqlite3 use qmark paramstyle - parameterized, never concatenated
print(f"target={TARGET} run_id={RUN_ID}")

target=local run_id=20260803T073010Z


## 1 — Schema DDL
T-SQL for Fabric SQL Database; the local run uses the portable subset. Note what the schema
encodes: connection **names** (not secrets), JSON for flexible table properties, an engine hint so
the orchestrator can route small entities to Python notebooks, and watermarks in their own table so
the hot read is a single row.

In [3]:
DDL_TSQL = """
CREATE TABLE dbo.etl_source (
  source_id        INT           NOT NULL PRIMARY KEY,
  source_name      NVARCHAR(100) NOT NULL UNIQUE,
  source_kind      NVARCHAR(20)  NOT NULL,     -- file | api | db
  connection_name  NVARCHAR(200) NULL,         -- Key Vault secret NAME, never the value
  base_location    NVARCHAR(400) NULL,
  default_format   NVARCHAR(20)  NOT NULL DEFAULT 'delta',
  enabled          BIT           NOT NULL DEFAULT 1
);

CREATE TABLE dbo.etl_entity (
  entity_id        INT           NOT NULL PRIMARY KEY,
  source_id        INT           NOT NULL REFERENCES dbo.etl_source(source_id),
  entity_name      NVARCHAR(200) NOT NULL,
  target_table     NVARCHAR(400) NOT NULL,
  layer            NVARCHAR(10)  NOT NULL,     -- bronze | silver | gold
  load_type        NVARCHAR(20)  NOT NULL,     -- full | incremental | cdf
  merge_keys       NVARCHAR(400) NULL,         -- comma separated
  watermark_column NVARCHAR(100) NULL,
  cluster_by       NVARCHAR(200) NULL,
  table_properties NVARCHAR(MAX) NULL,         -- JSON: delta.* properties
  engine_hint      NVARCHAR(20)  NOT NULL DEFAULT 'spark',   -- spark | python
  pool_hint        NVARCHAR(20)  NULL,
  enabled          BIT           NOT NULL DEFAULT 1,
  CONSTRAINT uq_entity UNIQUE (source_id, entity_name)
);
CREATE INDEX ix_entity_enabled ON dbo.etl_entity(enabled) INCLUDE (source_id, layer);

CREATE TABLE dbo.etl_watermark (
  entity_id        INT           NOT NULL PRIMARY KEY REFERENCES dbo.etl_entity(entity_id),
  watermark_value  NVARCHAR(100) NULL,
  updated_at       DATETIME2     NOT NULL DEFAULT SYSUTCDATETIME()
);

CREATE TABLE dbo.etl_run_log (
  run_log_id       BIGINT IDENTITY(1,1) PRIMARY KEY,
  run_id           NVARCHAR(40)  NOT NULL,
  entity_id        INT           NOT NULL REFERENCES dbo.etl_entity(entity_id),
  started_at       DATETIME2     NOT NULL,
  ended_at         DATETIME2     NULL,
  status           NVARCHAR(20)  NOT NULL,     -- OK | FAILED | SKIPPED
  rows_read        BIGINT        NULL,
  rows_written     BIGINT        NULL,
  watermark_from   NVARCHAR(100) NULL,
  watermark_to     NVARCHAR(100) NULL,
  spark_app_id     NVARCHAR(100) NULL,         -- the jump-off into the Monitoring hub
  error_message    NVARCHAR(2000) NULL
);
CREATE INDEX ix_runlog_entity ON dbo.etl_run_log(entity_id, started_at DESC);

CREATE TABLE dbo.dq_rules (
  rule_id          INT           NOT NULL PRIMARY KEY,
  entity_id        INT           NOT NULL REFERENCES dbo.etl_entity(entity_id),
  rule_kind        NVARCHAR(40)  NOT NULL,
  rule_params      NVARCHAR(MAX) NOT NULL,     -- JSON
  severity         NVARCHAR(10)  NOT NULL,     -- error | warn
  enabled          BIT           NOT NULL DEFAULT 1
);

CREATE VIEW dbo.vw_active_entities AS
SELECT e.entity_id, e.entity_name, e.target_table, e.layer, e.load_type, e.merge_keys,
       e.watermark_column, e.cluster_by, e.table_properties, e.engine_hint, e.pool_hint,
       s.source_name, s.source_kind, s.base_location, s.default_format, s.connection_name,
       w.watermark_value
FROM dbo.etl_entity e
JOIN dbo.etl_source s ON s.source_id = e.source_id
LEFT JOIN dbo.etl_watermark w ON w.entity_id = e.entity_id
WHERE e.enabled = 1 AND s.enabled = 1;
"""
print(DDL_TSQL[:400] + "\n... (full DDL above; deploy via SQL editor or a versioned .sql in git)")


CREATE TABLE dbo.etl_source (
  source_id        INT           NOT NULL PRIMARY KEY,
  source_name      NVARCHAR(100) NOT NULL UNIQUE,
  source_kind      NVARCHAR(20)  NOT NULL,     -- file | api | db
  connection_name  NVARCHAR(200) NULL,         -- Key Vault secret NAME, never the value
  base_location    NVARCHAR(400) NULL,
  default_format   NVARCHAR(20)  NOT NULL DEFAULT 'delta',
  enabled  
... (full DDL above; deploy via SQL editor or a versioned .sql in git)


In [4]:
# Local equivalent - same shapes, portable types, so every statement below really executes.
DDL_LOCAL = [
 """CREATE TABLE IF NOT EXISTS etl_source(source_id INTEGER PRIMARY KEY, source_name TEXT UNIQUE NOT NULL,
    source_kind TEXT NOT NULL, connection_name TEXT, base_location TEXT,
    default_format TEXT NOT NULL DEFAULT 'delta', enabled INTEGER NOT NULL DEFAULT 1)""",
 """CREATE TABLE IF NOT EXISTS etl_entity(entity_id INTEGER PRIMARY KEY,
    source_id INTEGER NOT NULL REFERENCES etl_source(source_id), entity_name TEXT NOT NULL,
    target_table TEXT NOT NULL, layer TEXT NOT NULL, load_type TEXT NOT NULL, merge_keys TEXT,
    watermark_column TEXT, cluster_by TEXT, table_properties TEXT,
    engine_hint TEXT NOT NULL DEFAULT 'spark', pool_hint TEXT, enabled INTEGER NOT NULL DEFAULT 1,
    UNIQUE(source_id, entity_name))""",
 "CREATE INDEX IF NOT EXISTS ix_entity_enabled ON etl_entity(enabled)",
 """CREATE TABLE IF NOT EXISTS etl_watermark(entity_id INTEGER PRIMARY KEY
    REFERENCES etl_entity(entity_id), watermark_value TEXT, updated_at TEXT NOT NULL)""",
 """CREATE TABLE IF NOT EXISTS etl_run_log(run_log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL, entity_id INTEGER NOT NULL REFERENCES etl_entity(entity_id),
    started_at TEXT NOT NULL, ended_at TEXT, status TEXT NOT NULL, rows_read INTEGER,
    rows_written INTEGER, watermark_from TEXT, watermark_to TEXT, spark_app_id TEXT, error_message TEXT)""",
 "CREATE INDEX IF NOT EXISTS ix_runlog_entity ON etl_run_log(entity_id, started_at DESC)",
 """CREATE TABLE IF NOT EXISTS dq_rules(rule_id INTEGER PRIMARY KEY, entity_id INTEGER NOT NULL
    REFERENCES etl_entity(entity_id), rule_kind TEXT NOT NULL, rule_params TEXT NOT NULL,
    severity TEXT NOT NULL, enabled INTEGER NOT NULL DEFAULT 1)""",
 """CREATE VIEW IF NOT EXISTS vw_active_entities AS
    SELECT e.entity_id, e.entity_name, e.target_table, e.layer, e.load_type, e.merge_keys,
           e.watermark_column, e.cluster_by, e.table_properties, e.engine_hint, e.pool_hint,
           s.source_name, s.source_kind, s.base_location, s.default_format, s.connection_name,
           w.watermark_value
    FROM etl_entity e JOIN etl_source s ON s.source_id = e.source_id
    LEFT JOIN etl_watermark w ON w.entity_id = e.entity_id
    WHERE e.enabled = 1 AND s.enabled = 1""",
]
if TARGET == "local" and os.path.exists(LOCAL_DB):
    os.remove(LOCAL_DB)
con = connect()
cur = con.cursor()
for stmt in (DDL_LOCAL if TARGET == "local" else [DDL_TSQL]):
    cur.execute(stmt)
con.commit()
print("schema created:", [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY name")] if TARGET=="local" else "fabric")

schema created: ['dq_rules', 'etl_entity', 'etl_run_log', 'etl_source', 'etl_watermark', 'sqlite_sequence', 'vw_active_entities']


## 2 — Seed configuration (parameterized inserts, never string concatenation)

In [5]:
sources = [(1, "erp_files", "file", "kv-erp-conn", "abfss://raw@onelake/erp", "parquet", 1),
           (2, "crm_api",   "api",  "kv-crm-conn", "https://crm.example/api", "json", 1)]
entities = [
 (1, 1, "orders",   "silver.orders",   "silver", "incremental", "order_id", "order_date",
  "customer_id,order_date", json.dumps({"delta.enableDeletionVectors":"true",
                                          "delta.enableChangeDataFeed":"true"}), "spark", "Medium", 1),
 (2, 1, "regions",  "bronze.regions",  "bronze", "full", None, None, None,
  json.dumps({}), "python", None, 1),
 (3, 2, "contacts", "bronze.contacts", "bronze", "incremental", "contact_id", "modified_at", None,
  json.dumps({"delta.enableChangeDataFeed":"true"}), "spark", "Small", 0),   # disabled
]
cur.executemany(f"INSERT INTO etl_source VALUES ({','.join([PARAM]*7)})", sources)
cur.executemany(f"INSERT INTO etl_entity VALUES ({','.join([PARAM]*13)})", entities)
cur.executemany(f"INSERT INTO dq_rules VALUES ({','.join([PARAM]*6)})", [
  (1, 1, "not_null",   json.dumps({"column":"customer_id"}), "error", 1),
  (2, 1, "min_rows",   json.dumps({"min": 100}),             "error", 1),
  (3, 2, "unique",     json.dumps({"column":"region_id"}),   "warn",  1),
])
con.commit()
print("seeded:", cur.execute("SELECT count(*) FROM etl_entity").fetchone()[0], "entities,",
      cur.execute("SELECT count(*) FROM dq_rules").fetchone()[0], "rules")

seeded: 3 entities, 3 rules


## 3 — The run pattern: fetch once, cache, work from memory, write twice
This is the performance shape that matters. The alternative — querying the metadata store inside
per-entity or per-partition code — turns the SQL DB into a bottleneck and can exhaust connections.

In [6]:
# ONE query per run. The view carries the enabled filters so notebooks embed no filter logic.
cur.execute("SELECT * FROM vw_active_entities ORDER BY entity_id")
cols = [d[0] for d in cur.description]
CONFIG = {r[cols.index("entity_id")]: dict(zip(cols, r)) for r in cur.fetchall()}
print(f"cached {len(CONFIG)} active entities (entity 3 correctly excluded - disabled)")
for eid, c in CONFIG.items():
    print(f"  {eid}: {c['entity_name']:<10} {c['layer']:<7} {c['load_type']:<12} "
          f"engine={c['engine_hint']:<6} wm={c['watermark_value']}")
assert 3 not in CONFIG, "disabled entity leaked into the active set"

cached 2 active entities (entity 3 correctly excluded - disabled)
  1: orders     silver  incremental  engine=spark  wm=None
  2: regions    bronze  full         engine=python wm=None


In [7]:
# Simulated run over the cached config; run-log rows ACCUMULATE, written once at the end.
run_rows, wm_updates = [], []
for eid, c in CONFIG.items():
    started = datetime.now(timezone.utc).isoformat()
    try:
        # ... real work would happen here, routed by c["engine_hint"] (Sec 21 engine choice) ...
        rows_read = 1500 if c["load_type"] == "incremental" else 300
        new_wm = date(2026, 8, 1).isoformat() if c["watermark_column"] else None
        run_rows.append((RUN_ID, eid, started, datetime.now(timezone.utc).isoformat(), "OK",
                         rows_read, rows_read, c["watermark_value"], new_wm, "app-local-demo", None))
        if new_wm:
            wm_updates.append((eid, new_wm))
    except Exception as ex:
        run_rows.append((RUN_ID, eid, started, datetime.now(timezone.utc).isoformat(), "FAILED",
                         None, None, c["watermark_value"], None, "app-local-demo", str(ex)[:2000]))

# WRITE 1: batched run log (one round trip, not one per row)
cur.executemany(f"""INSERT INTO etl_run_log
  (run_id, entity_id, started_at, ended_at, status, rows_read, rows_written,
   watermark_from, watermark_to, spark_app_id, error_message)
  VALUES ({','.join([PARAM]*11)})""", run_rows)

# WRITE 2: forward-only watermark upsert. T-SQL uses MERGE; the guard is the same either way -
# the watermark must never move backwards when two runs interleave.
UPSERT_TSQL = """
MERGE dbo.etl_watermark AS t
USING (SELECT ? AS entity_id, ? AS wm) AS s ON t.entity_id = s.entity_id
WHEN MATCHED AND (t.watermark_value IS NULL OR s.wm > t.watermark_value)
  THEN UPDATE SET watermark_value = s.wm, updated_at = SYSUTCDATETIME()
WHEN NOT MATCHED THEN INSERT (entity_id, watermark_value, updated_at) VALUES (s.entity_id, s.wm, SYSUTCDATETIME());
"""
for eid, wm in wm_updates:
    cur.execute(f"""INSERT INTO etl_watermark(entity_id, watermark_value, updated_at)
        VALUES ({PARAM},{PARAM},{PARAM})
        ON CONFLICT(entity_id) DO UPDATE SET watermark_value=excluded.watermark_value,
          updated_at=excluded.updated_at
        WHERE etl_watermark.watermark_value IS NULL
           OR excluded.watermark_value > etl_watermark.watermark_value""",
        (eid, wm, datetime.now(timezone.utc).isoformat()))
con.commit()
print(f"wrote {len(run_rows)} run-log rows in ONE batch; {len(wm_updates)} watermarks upserted")

wrote 2 run-log rows in ONE batch; 1 watermarks upserted


In [8]:
# Prove the forward-only guard: attempt to move a watermark BACKWARDS.
cur.execute(f"SELECT watermark_value FROM etl_watermark WHERE entity_id={PARAM}", (1,))
before = cur.fetchone()[0]
cur.execute(f"""INSERT INTO etl_watermark(entity_id, watermark_value, updated_at)
    VALUES ({PARAM},{PARAM},{PARAM})
    ON CONFLICT(entity_id) DO UPDATE SET watermark_value=excluded.watermark_value
    WHERE etl_watermark.watermark_value IS NULL
       OR excluded.watermark_value > etl_watermark.watermark_value""",
    (1, "2020-01-01", datetime.now(timezone.utc).isoformat()))
con.commit()
cur.execute(f"SELECT watermark_value FROM etl_watermark WHERE entity_id={PARAM}", (1,))
after = cur.fetchone()[0]
print(f"watermark before={before} after backwards attempt={after}")
assert before == after, "watermark moved backwards - the guard is broken"
print("PASS: watermark is forward-only under interleaved runs")

watermark before=2026-08-01 after backwards attempt=2026-08-01
PASS: watermark is forward-only under interleaved runs


In [9]:
# Operational queries the platform team actually needs.
print("--- last run per entity ---")
for r in cur.execute("""SELECT e.entity_name, l.status, l.rows_read, l.watermark_to, l.spark_app_id
                        FROM etl_run_log l JOIN etl_entity e ON e.entity_id = l.entity_id
                        WHERE l.run_id = ? ORDER BY e.entity_id""", (RUN_ID,)):
    print("  ", r)
print("--- entities never successfully run (candidates for alerting) ---")
print("  ", cur.execute("""SELECT count(*) FROM etl_entity e WHERE e.enabled = 1
                           AND NOT EXISTS (SELECT 1 FROM etl_run_log l
                                           WHERE l.entity_id = e.entity_id AND l.status='OK')""").fetchone()[0])
con.close()
print("\nconnection closed (always close in a finally block inside runMultiple activities)")

--- last run per entity ---
   ('orders', 'OK', 1500, '2026-08-01', 'app-local-demo')
   ('regions', 'OK', 300, None, 'app-local-demo')
--- entities never successfully run (candidates for alerting) ---
   0

connection closed (always close in a finally block inside runMultiple activities)


## 4 — Gotchas checklist (the executable notes)
1. **Token packing** — UTF-16-LE + length prefix into `attrs_before[1256]`; a subtle mistake yields an opaque login failure.
2. **ODBC driver** — probe `pyodbc.drivers()`; the image's driver version differs across Runtime 1.3 and 2.0.
3. **Never write run logs row-by-row via Spark JDBC** — use `executemany` (shown above) or accumulate and write once.
4. **Never query metadata inside a per-partition function** — fetch once, cache, broadcast if needed.
5. **Connections are not shareable across concurrent `runMultiple` activities** — open per activity, close in `finally`.
6. **Watermarks must be forward-only** — the guard is asserted above; last-writer-wins silently reprocesses or skips data.
7. **Parameterize everything** — security *and* plan-cache reuse.
8. **Secrets by name only** — `connection_name` holds a Key Vault secret name, never a value.
9. **Validate JSON columns on read** — `table_properties` is flexible and unvalidated; fail loudly on a typo.
10. **Mirroring is near-real-time, not synchronous** — control-plane reads go direct via pyodbc; analytical reads may use the OneLake mirror.
11. **Version the DDL in git** — the metadata schema is code; deploy it through the same dev→test→prod path as notebooks.